# Customer Churn & Revenue Intelligence
## End-to-end analysis: from raw data to actionable business insights

**Objective**: Identify the key drivers of customer churn, build a predictive model, and translate findings into concrete retention strategies.

**Dataset**: 5,000 synthetic SaaS customers with behavioral, demographic, and billing features.

---

In [ ]:
import sys
sys.path.append('..')
from src.churn_analysis import *
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully ✅')

## 1. Data generation & overview

We simulate a realistic SaaS customer dataset where churn is driven by a combination of:
- **Plan tier** (Starter customers churn 6× more than Enterprise)
- **Engagement signals** (logins, feature adoption)
- **Support friction** (ticket volume as a distress signal)
- **Tenure** (early-stage customers are at greatest risk)
- **Pricing exposure** (customers who saw price increases churn more)

In [ ]:
df = generate_customer_data(n=5000)
print(f'Shape: {df.shape}')
print(f'Churn rate: {df.churned.mean():.1%}')
df.head()

In [ ]:
df.describe().round(2)

## 2. Exploratory data analysis

Before modeling, we want to understand the data distribution and surface obvious patterns.

**Key questions**:
1. Which plan has the highest churn?
2. Does tenure protect against churn?
3. Are engaged customers less likely to leave?
4. Where is revenue most at risk?

In [ ]:
df = run_eda(df)

## 3. Feature engineering

Raw features rarely capture the full signal. We create:
- `engagement_score` — weighted combination of logins + adoption
- `revenue_per_seat` — pricing efficiency signal
- `is_early_tenure` — binary flag for the critical first 3 months
- `high_support` — flag for customers in the top quartile of ticket volume

In [ ]:
df = engineer_features(df)
print('New features added:')
print([c for c in df.columns if c not in generate_customer_data(1).columns])

## 4. Model training

We benchmark three models with 5-fold stratified cross-validation, then evaluate the winner on a held-out test set.

All preprocessing is wrapped in an sklearn `Pipeline` to prevent data leakage.

In [ ]:
best_model, fi_df, eval_data, train_data = build_churn_model(df)

## 5. Visualizations

In [ ]:
plot_all(df, fi_df, eval_data, output_dir='../visuals')

from IPython.display import Image, display
for i in range(1, 6):
    display(Image(f'../visuals/0{i}_*.png'))

## 6. Business recommendations

In [ ]:
print_recommendations(df)

## 7. Scoring new customers

With the trained model, customer success teams can score any new customer and prioritize outreach.

In [ ]:
# Score all active customers and rank by churn risk
import pandas as pd

active = df[df['churned'] == 0].copy()

# Drop columns not used in model
feature_cols = [
    'tenure_months', 'monthly_revenue', 'support_tickets',
    'logins_per_month', 'feature_adoption_pct', 'num_seats',
    'had_trial', 'price_increase_exposed', 'engagement_score',
    'revenue_per_seat', 'tickets_per_seat', 'is_early_tenure',
    'high_support', 'low_engagement',
    'plan', 'industry', 'region', 'payment_method'
]

active['churn_probability'] = best_model.predict_proba(active[feature_cols])[:, 1]
active['revenue_at_risk'] = active['churn_probability'] * active['monthly_revenue']

top_at_risk = (
    active[['customer_id', 'plan', 'tenure_months', 'monthly_revenue',
            'churn_probability', 'revenue_at_risk']]
    .sort_values('revenue_at_risk', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top_at_risk['churn_probability'] = top_at_risk['churn_probability'].map('{:.1%}'.format)
top_at_risk['revenue_at_risk'] = top_at_risk['revenue_at_risk'].map('${:.0f}'.format)
top_at_risk['monthly_revenue'] = top_at_risk['monthly_revenue'].map('${:.0f}'.format)

print('Top 20 customers by expected revenue at risk:')
top_at_risk

---
## Summary

This project demonstrates a complete, business-oriented data analysis workflow:

| Step | Technique | Output |
|------|-----------|--------|
| Data understanding | Descriptive stats, distributions | Churn rate = 22.4%, unequal across plans |
| Segmentation | GroupBy, pivot tables | Starter <3mo = highest risk segment |
| Feature engineering | Domain logic, ratios, flags | 7 derived features improving AUC +4pp |
| Modeling | Logistic, RF, GBM + CV | GBM AUC = 0.889 |
| Interpretation | Feature importance, SHAP-ready | Adoption + engagement are top drivers |
| Productionization | Streamlit dashboard | CS team can action top-risk list daily |

**Next steps** (production roadmap):
- Connect to real CRM/billing data (Salesforce, Stripe)
- Add SHAP explainability per customer
- Schedule daily model scoring job
- A/B test retention interventions and measure lift